# Build Autonomous Agent Prediction submission

This self-contained notebook reconstructs the validated Agent Config and creates `/kaggle/working/submission.zip`. No internet or dataset attachment is required.

In [ ]:
from pathlib import Path
import base64, json, shutil, zipfile

FILES = json.loads("{\"agent.yaml\": \"bmFtZTogcm9idXN0X3RhYnVsYXJfYXV0b21sCmRlc2NyaXB0aW9uOiBCdWRnZXQtYXdhcmUgYXV0b25vbW91cyBiaW5hcnkgdGFidWxhciBjbGFzc2lmaWNhdGlvbiBhZ2VudC4KbW9kZWw6IGdlbWluaS0zLjEtZmxhc2gtbGl0ZQppbnN0cnVjdGlvbjogIWluY2x1ZGUgcHJvbXB0cy9zeXN0ZW0ubWQKdG9vbHM6CiAgLSBydW5fY29tbWFuZAogIC0gc3VibWl0X3ByZWRpY3Rpb25zCiAgLSBzZWxlY3Rfc3VibWlzc2lvbgogIC0gZ2V0X3N0YXR1cwpza2lsbHM6CiAgLSBza2lsbHMvdGFidWxhci1hdXRvbWwKZ2VuZXJhdGVfY29udGVudF9jb25maWc6ICFpbmNsdWRlIGNvbmZpZ3Mvc2FtcGxpbmcueWFtbAo=\", \"configs/sampling.yaml\": \"dGVtcGVyYXR1cmU6IDAuMQptYXhfb3V0cHV0X3Rva2VuczogNDA5Ngp0aGlua2luZ19jb25maWc6CiAgdGhpbmtpbmdfYnVkZ2V0OiAxMDI0CiAgaW5jbHVkZV90aG91Z2h0czogZmFsc2UK\", \"prompts/system.md\": \"WW91IGFyZSBhIGRpc2NpcGxpbmVkIGF1dG9ub21vdXMgbWFjaGluZS1sZWFybmluZyBjb21wZXRpdG9yLiBDb21wbGV0ZSB0aGUgYmluYXJ5IHRhYnVsYXIgdGFzaywgbWF4aW1pemUge21ldHJpY19uYW1lfSAoe21ldHJpY19kaXJlY3Rpb259KSwgYW5kIGZpbmlzaCBieSBzZWxlY3RpbmcgZXhhY3RseSB0d28gcm9idXN0IHN1Ym1pc3Npb25zLgoKIyMgUnVudGltZSBjb250ZXh0Cgp7cHJvYmxlbV9kZXNjcmlwdGlvbn0KClRoZSB3b3JraW5nIGRpcmVjdG9yeSBjb250YWlucyBgdHJhaW4uY3N2YCwgYHRlc3QuY3N2YCwgYW5kIGBzYW1wbGVfc3VibWlzc2lvbi5jc3ZgLiBUaGUgTGludXggc2FuZGJveCBpcyBvZmZsaW5lIGJ1dCBpbmNsdWRlcyBwYW5kYXMsIE51bVB5LCBzY2lraXQtbGVhcm4sIENhdEJvb3N0LCBMaWdodEdCTSwgWEdCb29zdCwgU2NpUHksIGFuZCBzdGFuZGFyZCBLYWdnbGUgcGFja2FnZXMuCgpIYXJkIGxpbWl0czoge21heF90aW1lX21pbnV0ZXN9IG1pbnV0ZXMsIHttYXhfc3VibWlzc2lvbnN9IHN1Ym1pc3Npb25zLCB7bWF4X3NlbGVjdGlvbnN9IHNlbGVjdGlvbnMsIHttYXhfdG9vbF9jYWxsc30gdG9vbCBjYWxscywge21heF9sbG1fY2FsbHN9IExMTSBjYWxscywgYW5kICR7bWF4X2J1ZGdldF91c2R9IHRvdGFsIG1vZGVsIGNvc3QuCgojIyBNYW5kYXRvcnkgd29ya2Zsb3cKCjEuIENhbGwgYGdldF9zdGF0dXNgIG9uY2UuCjIuIFVzZSB0aGUgYHRhYnVsYXItYXV0b21sYCBza2lsbCBpbW1lZGlhdGVseS4gUnVuIGBzY3JpcHRzL2F1dG9tbC5weWAgd2l0aCB0aGUgc2tpbGwtc2NyaXB0IHRvb2wuIERvIG5vdCByZWltcGxlbWVudCBpdHMgbW9kZWxpbmcgbG9naWMgYW5kIGRvIG5vdCBwZXJmb3JtIG9wZW4tZW5kZWQgRURBLiBUaGUgc2NyaXB0IGluc3BlY3RzIHRoZSBzY2hlbWEsIHBlcmZvcm1zIGNyb3NzLXZhbGlkYXRpb24sIHRyYWlucyBhIGRpdmVyc2UgcG9ydGZvbGlvLCBhbmQgd3JpdGVzIGNhbmRpZGF0ZSBzdWJtaXNzaW9uIENTVnMgcGx1cyBgYXV0b21sX21hbmlmZXN0Lmpzb25gLgozLiBJbnNwZWN0IHRoZSBzY3JpcHQncyBjb25jaXNlIHN0ZG91dCBvciBgYXV0b21sX21hbmlmZXN0Lmpzb25gLiBDYW5kaWRhdGUgZmlsZXMgYXJlIG9yZGVyZWQgYnkgY3Jvc3MtdmFsaWRhdGVkIEFVQy4gU3VibWl0IG5vIG1vcmUgdGhhbiB0aGUgZmlyc3Qgc2l4IGRpc3RpbmN0IGNhbmRpZGF0ZXMgdXNpbmcgYHN1Ym1pdF9wcmVkaWN0aW9uc2AuCjQuIFRyZWF0IHB1YmxpYyBzY29yZXMgYXMgbm9pc3kgZXN0aW1hdGVzIGZyb20gb25seSBoYWxmIHRoZSB0ZXN0IHNldC4gRG8gbm90IHR1bmUgcHJlZGljdGlvbiB2YWx1ZXMgb3IgcmVwZWF0ZWRseSBnZW5lcmF0ZSB2YXJpYW50cyBhZ2FpbnN0IHRoZSBsZWFkZXJib2FyZC4gU3VibWl0IGVhY2ggcHJlY29tcHV0ZWQgY2FuZGlkYXRlIGF0IG1vc3Qgb25jZS4KNS4gU2VsZWN0IGV4YWN0bHkgdHdvIHN1Ym1pc3Npb25zOiB0aGUgYmVzdCBwdWJsaWMgc2NvcmVyIGFuZCB0aGUgc3Ryb25nZXN0IG1lYW5pbmdmdWxseSBkaWZmZXJlbnQgY2FuZGlkYXRlLiBQcmVmZXIgdGhlIG1hbmlmZXN0J3MgcmVjb21tZW5kZWQgZGl2ZXJzZSBjYW5kaWRhdGUgd2hlbiBpdHMgcHVibGljIHNjb3JlIGlzIHdpdGhpbiAwLjAxIG9mIHRoZSBiZXN0OyBvdGhlcndpc2UgY2hvb3NlIHRoZSBzZWNvbmQtYmVzdCBwdWJsaWMgc2NvcmVyLiBOZXZlciBzZWxlY3QgZHVwbGljYXRlIHByZWRpY3Rpb25zLgo2LiBDYWxsIGBnZXRfc3RhdHVzYCwgdGhlbiBgc2VsZWN0X3N1Ym1pc3Npb25gIHdpdGggdGhlIHR3byBJRHMuIEVuZCBpbW1lZGlhdGVseSBhZnRlciBzZWxlY3Rpb24uCgojIyBGYWlsdXJlIHJlY292ZXJ5CgpJZiB0aGUgZnVsbCBzY3JpcHQgZmFpbHMsIHJlYWQgaXRzIGVycm9yLCBmaXggb25seSB0aGUgZGlyZWN0IGNvbXBhdGliaWxpdHkgaXNzdWUsIGFuZCByZXJ1biBvbmNlIHdpdGggYC0tZmFzdGAuIElmIHRoYXQgYWxzbyBmYWlscywgcnVuIGBzY3JpcHRzL2F1dG9tbC5weSAtLWZhbGxiYWNrYCwgc3VibWl0IGl0cyBvdXRwdXRzLCBzZWxlY3QgdGhlIHR3byBiZXN0IGRpc3RpbmN0IHN1Ym1pc3Npb25zLCBhbmQgZmluaXNoLiBBbHdheXMgcHJlc2VydmUgdGltZSBmb3IgZmluYWwgc3VibWlzc2lvbiBzZWxlY3Rpb24uCg==\", \"skills/tabular-automl/SKILL.md\": \"LS0tCm5hbWU6IHRhYnVsYXItYXV0b21sCmRlc2NyaXB0aW9uOiBSdW5zIGEgcHJlLXRlc3RlZCwgYnVkZ2V0LWF3YXJlIG1vZGVsIHBvcnRmb2xpbyBmb3IgbWl4ZWQtdHlwZSBiaW5hcnkgdGFidWxhciBjbGFzc2lmaWNhdGlvbiBhbmQgcHJvZHVjZXMgcmFua2VkIHN1Ym1pc3Npb24gY2FuZGlkYXRlcy4KLS0tCgojIFRhYnVsYXIgQXV0b01MCgpVc2UgdGhpcyBza2lsbCBleGFjdGx5IG9uY2UgYXQgdGhlIGJlZ2lubmluZyBvZiBhIGJpbmFyeSBjbGFzc2lmaWNhdGlvbiB0YXNrLgoKIyMgU2NyaXB0CgpSdW4gYHNjcmlwdHMvYXV0b21sLnB5YCBpbiB0aGUgc2FuZGJveCB3b3JraW5nIGRpcmVjdG9yeS4gSXQgYXV0b21hdGljYWxseToKCi0gaW5mZXJzIHRoZSB0YXJnZXQgYW5kIGlkZW50aWZpZXIgZnJvbSB0aGUgc3VwcGxpZWQgQ1NWIGZpbGVzOwotIGhhbmRsZXMgbnVtZXJpY2FsLCBjYXRlZ29yaWNhbCwgb3JkaW5hbCwgYW5kIG1pc3NpbmcgdmFsdWVzOwotIGNyb3NzLXZhbGlkYXRlcyBDYXRCb29zdCwgTGlnaHRHQk0sIEV4dHJhVHJlZXMsIGFuZCByZWd1bGFyaXplZCBsaW5lYXIgbW9kZWxzOwotIGNyZWF0ZXMgbGVha2FnZS1zYWZlIG91dC1vZi1mb2xkIHByZWRpY3Rpb25zOwotIGJ1aWxkcyByb2J1c3QgcmFuayBlbnNlbWJsZXMgd2l0aG91dCB1c2luZyB0ZXN0IGxhYmVsczsKLSB3cml0ZXMgYGNhbmRpZGF0ZV8qLmNzdmAgZmlsZXMgbWF0Y2hpbmcgYHNhbXBsZV9zdWJtaXNzaW9uLmNzdmAgZXhhY3RseTsKLSB3cml0ZXMgYGF1dG9tbF9tYW5pZmVzdC5qc29uYCB3aXRoIENWIHNjb3JlcywgZmlsZSBvcmRlciwgZGl2ZXJzaXR5LCBhbmQgcmVjb21tZW5kYXRpb25zLgoKVXNlIGAtLWZhc3RgIG9ubHkgYWZ0ZXIgYSBub3JtYWwgcnVuIGZhaWxzIG9yIHRoZSByZW1haW5pbmcgcnVudGltZSBpcyB1bmRlciAyMCBtaW51dGVzLiBVc2UgYC0tZmFsbGJhY2tgIG9ubHkgaWYgb3B0aW9uYWwgYm9vc3RpbmcgbGlicmFyaWVzIGZhaWwuCgpTdWJtaXQgYXQgbW9zdCB0aGUgZmlyc3Qgc2l4IGZpbGVzIGxpc3RlZCBpbiB0aGUgbWFuaWZlc3QuIFB1YmxpYyBsZWFkZXJib2FyZCBmZWVkYmFjayBpcyBmb3IgY29hcnNlIG1vZGVsIHNlbGVjdGlvbiBvbmx5LCBuZXZlciBwcmVkaWN0aW9uLWxldmVsIHR1bmluZy4K\", \"skills/tabular-automl/scripts/automl.py\": \"IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJCdWRnZXQtYXdhcmUgbWl4ZWQtdHlwZSBBdXRvTUwgZm9yIHRoZSBLYWdnbGUtaW4tS2FnZ2xlIHNhbmRib3guIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCB0aW1lCmltcG9ydCB3YXJuaW5ncwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCmZyb20gc2NpcHkuc3RhdHMgaW1wb3J0IHJhbmtkYXRhCmZyb20gc2tsZWFybi5iYXNlIGltcG9ydCBjbG9uZQpmcm9tIHNrbGVhcm4uY29tcG9zZSBpbXBvcnQgQ29sdW1uVHJhbnNmb3JtZXIKZnJvbSBza2xlYXJuLmVuc2VtYmxlIGltcG9ydCBFeHRyYVRyZWVzQ2xhc3NpZmllciwgUmFuZG9tRm9yZXN0Q2xhc3NpZmllcgpmcm9tIHNrbGVhcm4uaW1wdXRlIGltcG9ydCBTaW1wbGVJbXB1dGVyCmZyb20gc2tsZWFybi5saW5lYXJfbW9kZWwgaW1wb3J0IExvZ2lzdGljUmVncmVzc2lvbgpmcm9tIHNrbGVhcm4ubWV0cmljcyBpbXBvcnQgcm9jX2F1Y19zY29yZQpmcm9tIHNrbGVhcm4ubW9kZWxfc2VsZWN0aW9uIGltcG9ydCBTdHJhdGlmaWVkS0ZvbGQKZnJvbSBza2xlYXJuLnBpcGVsaW5lIGltcG9ydCBQaXBlbGluZQpmcm9tIHNrbGVhcm4ucHJlcHJvY2Vzc2luZyBpbXBvcnQgT25lSG90RW5jb2RlciwgT3JkaW5hbEVuY29kZXIsIFN0YW5kYXJkU2NhbGVyCgp3YXJuaW5ncy5maWx0ZXJ3YXJuaW5ncygiaWdub3JlIikKU0VFRCA9IDIwMjYwNzE3CgoKZGVmIHJhbmswMSh2YWx1ZXM6IG5wLm5kYXJyYXkpIC0+IG5wLm5kYXJyYXk6CiAgICB2YWx1ZXMgPSBucC5hc2FycmF5KHZhbHVlcywgZHR5cGU9ZmxvYXQpCiAgICByZXR1cm4gcmFua2RhdGEodmFsdWVzLCBtZXRob2Q9ImF2ZXJhZ2UiKSAvIChsZW4odmFsdWVzKSArIDEuMCkKCgpkZWYgZmluZF9jb2x1bW5zKHRyYWluOiBwZC5EYXRhRnJhbWUsIHRlc3Q6IHBkLkRhdGFGcmFtZSwgc2FtcGxlOiBwZC5EYXRhRnJhbWUpOgogICAgdGFyZ2V0X2NhbmRpZGF0ZXMgPSBbYyBmb3IgYyBpbiB0cmFpbi5jb2x1bW5zIGlmIGMgbm90IGluIHRlc3QuY29sdW1uc10KICAgIGlmIGxlbih0YXJnZXRfY2FuZGlkYXRlcykgIT0gMToKICAgICAgICB0YXJnZXRfY2FuZGlkYXRlcyA9IFtjIGZvciBjIGluIHNhbXBsZS5jb2x1bW5zIGlmIGMgbm90IGluIHRlc3QuY29sdW1ucyBvciBjIGluIHRyYWluLmNvbHVtbnNdCiAgICB0YXJnZXQgPSAidGFyZ2V0IiBpZiAidGFyZ2V0IiBpbiB0YXJnZXRfY2FuZGlkYXRlcyBlbHNlIHRhcmdldF9jYW5kaWRhdGVzWy0xXQogICAgcHJlZF9jb2xzID0gW2MgZm9yIGMgaW4gc2FtcGxlLmNvbHVtbnMgaWYgYyAhPSB0YXJnZXRdCiAgICBpZF9jb2wgPSBwcmVkX2NvbHNbMF0gaWYgcHJlZF9jb2xzIGVsc2UgTm9uZQogICAgZmVhdHVyZXMgPSBbYyBmb3IgYyBpbiB0ZXN0LmNvbHVtbnMgaWYgYyAhPSBpZF9jb2xdCiAgICByZXR1cm4gdGFyZ2V0LCBpZF9jb2wsIGZlYXR1cmVzCgoKZGVmIG5vcm1hbGl6ZV90YXJnZXQoc2VyaWVzOiBwZC5TZXJpZXMpOgogICAgdmFscyA9IGxpc3QocGQuU2VyaWVzKHNlcmllcy5kcm9wbmEoKS51bmlxdWUoKSkuc29ydF92YWx1ZXMoKSkKICAgIGlmIGxlbih2YWxzKSAhPSAyOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJFeHBlY3RlZCBhIGJpbmFyeSB0YXJnZXQsIGZvdW5kIHt2YWxzfSIpCiAgICBtYXBwaW5nID0ge3ZhbHNbMF06IDAsIHZhbHNbMV06IDF9CiAgICByZXR1cm4gc2VyaWVzLm1hcChtYXBwaW5nKS5hc3R5cGUoaW50KS50b19udW1weSgpLCBtYXBwaW5nCgoKZGVmIHByZXBhcmVfZnJhbWVzKHRyYWluLCB0ZXN0LCBmZWF0dXJlcyk6CiAgICB4dHIgPSB0cmFpbltmZWF0dXJlc10uY29weSgpCiAgICB4dGUgPSB0ZXN0W2ZlYXR1cmVzXS5jb3B5KCkKICAgIGNhdF9jb2xzID0gW10KICAgIGZvciBjb2wgaW4gZmVhdHVyZXM6CiAgICAgICAgY29tYmluZWQgPSBwZC5jb25jYXQoW3h0cltjb2xdLCB4dGVbY29sXV0sIGlnbm9yZV9pbmRleD1UcnVlKQogICAgICAgIGlzX2NhdCA9ICgKICAgICAgICAgICAgbm90IHBkLmFwaS50eXBlcy5pc19udW1lcmljX2R0eXBlKGNvbWJpbmVkKQogICAgICAgICAgICBvciBwZC5hcGkudHlwZXMuaXNfYm9vbF9kdHlwZShjb21iaW5lZCkKICAgICAgICAgICAgb3IgKGNvbWJpbmVkLm51bmlxdWUoZHJvcG5hPVRydWUpIDw9IDIwIGFuZCBub3QgcGQuYXBpLnR5cGVzLmlzX2Zsb2F0X2R0eXBlKGNvbWJpbmVkKSkKICAgICAgICApCiAgICAgICAgaWYgaXNfY2F0OgogICAgICAgICAgICBjYXRfY29scy5hcHBlbmQoY29sKQogICAgICAgICAgICB4dHJbY29sXSA9IHh0cltjb2xdLmFzdHlwZSgic3RyaW5nIikuZmlsbG5hKCJfX01JU1NJTkdfXyIpCiAgICAgICAgICAgIHh0ZVtjb2xdID0geHRlW2NvbF0uYXN0eXBlKCJzdHJpbmciKS5maWxsbmEoIl9fTUlTU0lOR19fIikKICAgICAgICBlbHNlOgogICAgICAgICAgICB4dHJbY29sXSA9IHBkLnRvX251bWVyaWMoeHRyW2NvbF0sIGVycm9ycz0iY29lcmNlIikKICAgICAgICAgICAgeHRlW2NvbF0gPSBwZC50b19udW1lcmljKHh0ZVtjb2xdLCBlcnJvcnM9ImNvZXJjZSIpCiAgICBudW1fY29scyA9IFtjIGZvciBjIGluIGZlYXR1cmVzIGlmIGMgbm90IGluIGNhdF9jb2xzXQogICAgcmV0dXJuIHh0ciwgeHRlLCBjYXRfY29scywgbnVtX2NvbHMKCgpkZWYgc2tsZWFybl9tb2RlbHMoY2F0X2NvbHMsIG51bV9jb2xzLCBuX3Jvd3MsIGZhbGxiYWNrPUZhbHNlKToKICAgIG9yZGluYWwgPSBDb2x1bW5UcmFuc2Zvcm1lcihbCiAgICAgICAgKCJudW0iLCBTaW1wbGVJbXB1dGVyKHN0cmF0ZWd5PSJtZWRpYW4iLCBhZGRfaW5kaWNhdG9yPVRydWUpLCBudW1fY29scyksCiAgICAgICAgKCJjYXQiLCBQaXBlbGluZShbCiAgICAgICAgICAgICgiaW1wIiwgU2ltcGxlSW1wdXRlcihzdHJhdGVneT0ibW9zdF9mcmVxdWVudCIpKSwKICAgICAgICAgICAgKCJlbmMiLCBPcmRpbmFsRW5jb2RlcihoYW5kbGVfdW5rbm93bj0idXNlX2VuY29kZWRfdmFsdWUiLCB1bmtub3duX3ZhbHVlPS0xKSksCiAgICAgICAgXSksIGNhdF9jb2xzKSwKICAgIF0sIHJlbWFpbmRlcj0iZHJvcCIpCiAgICB0cmVlcyA9IDUwMCBpZiBuX3Jvd3MgPCAyMDAwMCBlbHNlIDM1MAogICAgcmVzdWx0ID0gewogICAgICAgICJleHRyYV90cmVlcyI6IFBpcGVsaW5lKFsKICAgICAgICAgICAgKCJwcmVwIiwgb3JkaW5hbCksCiAgICAgICAgICAgICgibW9kZWwiLCBFeHRyYVRyZWVzQ2xhc3NpZmllcigKICAgICAgICAgICAgICAgIG5fZXN0aW1hdG9ycz10cmVlcywgbWluX3NhbXBsZXNfbGVhZj1tYXgoMSwgaW50KG5wLnNxcnQobl9yb3dzKSAvIDM1KSksCiAgICAgICAgICAgICAgICBtYXhfZmVhdHVyZXM9InNxcnQiLCBjbGFzc193ZWlnaHQ9ImJhbGFuY2VkIiwgbl9qb2JzPS0xLCByYW5kb21fc3RhdGU9U0VFRCwKICAgICAgICAgICAgKSksCiAgICAgICAgXSkKICAgIH0KICAgIGlmIGZhbGxiYWNrOgogICAgICAgIHJlc3VsdFsicmFuZG9tX2ZvcmVzdCJdID0gUGlwZWxpbmUoWwogICAgICAgICAgICAoInByZXAiLCBjbG9uZShvcmRpbmFsKSksCiAgICAgICAgICAgICgibW9kZWwiLCBSYW5kb21Gb3Jlc3RDbGFzc2lmaWVyKAogICAgICAgICAgICAgICAgbl9lc3RpbWF0b3JzPXRyZWVzLCBtaW5fc2FtcGxlc19sZWFmPW1heCgyLCBpbnQobnAuc3FydChuX3Jvd3MpIC8gMjUpKSwKICAgICAgICAgICAgICAgIG1heF9mZWF0dXJlcz0wLjcsIGNsYXNzX3dlaWdodD0iYmFsYW5jZWRfc3Vic2FtcGxlIiwgbl9qb2JzPS0xLCByYW5kb21fc3RhdGU9U0VFRCArIDEsCiAgICAgICAgICAgICkpLAogICAgICAgIF0pCiAgICBpZiBuX3Jvd3MgPD0gMzAwMDA6CiAgICAgICAgb25laG90ID0gQ29sdW1uVHJhbnNmb3JtZXIoWwogICAgICAgICAgICAoIm51bSIsIFBpcGVsaW5lKFsoImltcCIsIFNpbXBsZUltcHV0ZXIoc3RyYXRlZ3k9Im1lZGlhbiIsIGFkZF9pbmRpY2F0b3I9VHJ1ZSkpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAoInNjYWxlIiwgU3RhbmRhcmRTY2FsZXIoKSldKSwgbnVtX2NvbHMpLAogICAgICAgICAgICAoImNhdCIsIE9uZUhvdEVuY29kZXIoaGFuZGxlX3Vua25vd249Imlnbm9yZSIsIG1pbl9mcmVxdWVuY3k9MiksIGNhdF9jb2xzKSwKICAgICAgICBdKQogICAgICAgIHJlc3VsdFsibG9naXN0aWMiXSA9IFBpcGVsaW5lKFsKICAgICAgICAgICAgKCJwcmVwIiwgb25laG90KSwKICAgICAgICAgICAgKCJtb2RlbCIsIExvZ2lzdGljUmVncmVzc2lvbihDPTAuMzUsIG1heF9pdGVyPTgwMCwgY2xhc3Nfd2VpZ2h0PSJiYWxhbmNlZCIsIG5fam9icz0tMSkpLAogICAgICAgIF0pCiAgICByZXR1cm4gcmVzdWx0CgoKZGVmIGFkZF9ib29zdGVycyhtb2RlbHMsIGNhdF9jb2xzLCBuX3Jvd3MsIGZhc3QpOgogICAgdHJ5OgogICAgICAgIGZyb20gY2F0Ym9vc3QgaW1wb3J0IENhdEJvb3N0Q2xhc3NpZmllcgogICAgICAgIGl0ZXJhdGlvbnMgPSA0NTAgaWYgZmFzdCBlbHNlICg3NTAgaWYgbl9yb3dzIDwgMjUwMDAgZWxzZSA1NTApCiAgICAgICAgbW9kZWxzWyJjYXRib29zdF9kNiJdID0gQ2F0Qm9vc3RDbGFzc2lmaWVyKAogICAgICAgICAgICBpdGVyYXRpb25zPWl0ZXJhdGlvbnMsIGRlcHRoPTYsIGxlYXJuaW5nX3JhdGU9MC4wNTUsIGxvc3NfZnVuY3Rpb249IkxvZ2xvc3MiLAogICAgICAgICAgICBldmFsX21ldHJpYz0iQVVDIiwgbDJfbGVhZl9yZWc9NSwgcmFuZG9tX3NlZWQ9U0VFRCwgdmVyYm9zZT1GYWxzZSwKICAgICAgICAgICAgYWxsb3dfd3JpdGluZ19maWxlcz1GYWxzZSwgdGhyZWFkX2NvdW50PS0xLAogICAgICAgICkKICAgICAgICBpZiBub3QgZmFzdDoKICAgICAgICAgICAgbW9kZWxzWyJjYXRib29zdF9kOCJdID0gQ2F0Qm9vc3RDbGFzc2lmaWVyKAogICAgICAgICAgICAgICAgaXRlcmF0aW9ucz1tYXgoNTAwLCBpdGVyYXRpb25zIC0gMTAwKSwgZGVwdGg9OCwgbGVhcm5pbmdfcmF0ZT0wLjA0LAogICAgICAgICAgICAgICAgbG9zc19mdW5jdGlvbj0iTG9nbG9zcyIsIGV2YWxfbWV0cmljPSJBVUMiLCBsMl9sZWFmX3JlZz04LAogICAgICAgICAgICAgICAgcmFuZG9tX3NlZWQ9U0VFRCArIDExLCB2ZXJib3NlPUZhbHNlLCBhbGxvd193cml0aW5nX2ZpbGVzPUZhbHNlLCB0aHJlYWRfY291bnQ9LTEsCiAgICAgICAgICAgICkKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOgogICAgICAgIHByaW50KGYiSU5GTyBDYXRCb29zdCB1bmF2YWlsYWJsZToge2V4Y30iKQogICAgdHJ5OgogICAgICAgIGZyb20gbGlnaHRnYm0gaW1wb3J0IExHQk1DbGFzc2lmaWVyCiAgICAgICAgbGVhdmVzID0gMTUgaWYgbl9yb3dzIDwgMjAwMCBlbHNlIDMxCiAgICAgICAgbW9kZWxzWyJsaWdodGdibSJdID0gTEdCTUNsYXNzaWZpZXIoCiAgICAgICAgICAgIG5fZXN0aW1hdG9ycz00NTAgaWYgZmFzdCBlbHNlIDc1MCwgbGVhcm5pbmdfcmF0ZT0wLjAzNSwKICAgICAgICAgICAgbnVtX2xlYXZlcz1sZWF2ZXMsIG1heF9kZXB0aD0tMSwgbWluX2NoaWxkX3NhbXBsZXM9bWF4KDEyLCBpbnQobnAuc3FydChuX3Jvd3MpKSksCiAgICAgICAgICAgIHN1YnNhbXBsZT0wLjg1LCBjb2xzYW1wbGVfYnl0cmVlPTAuODUsIHJlZ19hbHBoYT0wLjIsIHJlZ19sYW1iZGE9Mi4wLAogICAgICAgICAgICByYW5kb21fc3RhdGU9U0VFRCArIDIzLCBuX2pvYnM9LTEsIHZlcmJvc2l0eT0tMSwKICAgICAgICApCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoKICAgICAgICBwcmludChmIklORk8gTGlnaHRHQk0gdW5hdmFpbGFibGU6IHtleGN9IikKCgpkZWYgZW5jb2RlZF9mb3JfbGdibSh4dHIsIHh0ZSwgY2F0X2NvbHMpOgogICAgYSA9IHh0ci5jb3B5KCkKICAgIGIgPSB4dGUuY29weSgpCiAgICBmb3IgY29sIGluIGNhdF9jb2xzOgogICAgICAgIGNhdGVnb3JpZXMgPSBwZC5JbmRleChwZC5jb25jYXQoW2FbY29sXSwgYltjb2xdXSwgaWdub3JlX2luZGV4PVRydWUpLmFzdHlwZShzdHIpLnVuaXF1ZSgpKQogICAgICAgIG1hcHBpbmcgPSBwZC5TZXJpZXMobnAuYXJhbmdlKGxlbihjYXRlZ29yaWVzKSksIGluZGV4PWNhdGVnb3JpZXMpCiAgICAgICAgYVtjb2xdID0gYVtjb2xdLmFzdHlwZShzdHIpLm1hcChtYXBwaW5nKS5hc3R5cGUoImludDMyIikKICAgICAgICBiW2NvbF0gPSBiW2NvbF0uYXN0eXBlKHN0cikubWFwKG1hcHBpbmcpLmFzdHlwZSgiaW50MzIiKQogICAgcmV0dXJuIGEsIGIKCgpkZWYgZml0X3ByZWRpY3RfbW9kZWwobmFtZSwgbW9kZWwsIHh0ciwgeHRlLCB5LCBmb2xkcywgY2F0X2NvbHMpOgogICAgb29mID0gbnAuemVyb3MobGVuKHh0ciksIGR0eXBlPWZsb2F0KQogICAgcHJlZCA9IG5wLnplcm9zKGxlbih4dGUpLCBkdHlwZT1mbG9hdCkKICAgIGZvbGRfc2NvcmVzID0gW10KICAgIGlzX2NhdGJvb3N0ID0gbmFtZS5zdGFydHN3aXRoKCJjYXRib29zdCIpCiAgICBpc19sZ2JtID0gbmFtZSA9PSAibGlnaHRnYm0iCiAgICBpZiBpc19sZ2JtOgogICAgICAgIHh0cl91c2UsIHh0ZV91c2UgPSBlbmNvZGVkX2Zvcl9sZ2JtKHh0ciwgeHRlLCBjYXRfY29scykKICAgIGVsc2U6CiAgICAgICAgeHRyX3VzZSwgeHRlX3VzZSA9IHh0ciwgeHRlCiAgICBmb3IgZm9sZCwgKGl0ciwgaXZhKSBpbiBlbnVtZXJhdGUoZm9sZHMpOgogICAgICAgIGZpdHRlZCA9IGNsb25lKG1vZGVsKQogICAgICAgIGZpdF9rd2FyZ3MgPSB7fQogICAgICAgIGlmIGlzX2NhdGJvb3N0OgogICAgICAgICAgICBmaXRfa3dhcmdzID0geyJjYXRfZmVhdHVyZXMiOiBjYXRfY29scywgImV2YWxfc2V0IjogKHh0cl91c2UuaWxvY1tpdmFdLCB5W2l2YV0pLAogICAgICAgICAgICAgICAgICAgICAgICAgICJlYXJseV9zdG9wcGluZ19yb3VuZHMiOiA4MCwgInZlcmJvc2UiOiBGYWxzZX0KICAgICAgICBlbGlmIGlzX2xnYm06CiAgICAgICAgICAgIGZpdF9rd2FyZ3MgPSB7ImNhdGVnb3JpY2FsX2ZlYXR1cmUiOiBjYXRfY29sc30KICAgICAgICBmaXR0ZWQuZml0KHh0cl91c2UuaWxvY1tpdHJdLCB5W2l0cl0sICoqZml0X2t3YXJncykKICAgICAgICBvb2ZbaXZhXSA9IGZpdHRlZC5wcmVkaWN0X3Byb2JhKHh0cl91c2UuaWxvY1tpdmFdKVs6LCAxXQogICAgICAgIHByZWQgKz0gZml0dGVkLnByZWRpY3RfcHJvYmEoeHRlX3VzZSlbOiwgMV0gLyBsZW4oZm9sZHMpCiAgICAgICAgZm9sZF9zY29yZXMuYXBwZW5kKHJvY19hdWNfc2NvcmUoeVtpdmFdLCBvb2ZbaXZhXSkpCiAgICByZXR1cm4gb29mLCBwcmVkLCBmb2xkX3Njb3JlcwoKCmRlZiBncmVlZHlfYmxlbmQob29mcywgcHJlZHMsIHksIG9yZGVyZWRfbmFtZXMpOgogICAgYmVzdCA9IG9yZGVyZWRfbmFtZXNbMF0KICAgIGJsZW5kX29vZiA9IHJhbmswMShvb2ZzW2Jlc3RdKQogICAgYmxlbmRfcHJlZCA9IHJhbmswMShwcmVkc1tiZXN0XSkKICAgIG1lbWJlcnMgPSBbYmVzdF0KICAgIGJlc3Rfc2NvcmUgPSByb2NfYXVjX3Njb3JlKHksIGJsZW5kX29vZikKICAgIGZvciBuYW1lIGluIG9yZGVyZWRfbmFtZXNbMTpdOgogICAgICAgIGNhbmRpZGF0ZV9vb2YgPSAwLjc1ICogYmxlbmRfb29mICsgMC4yNSAqIHJhbmswMShvb2ZzW25hbWVdKQogICAgICAgIHNjb3JlID0gcm9jX2F1Y19zY29yZSh5LCBjYW5kaWRhdGVfb29mKQogICAgICAgIGlmIHNjb3JlID49IGJlc3Rfc2NvcmUgLSAwLjAwMDM6CiAgICAgICAgICAgIGJsZW5kX29vZiA9IGNhbmRpZGF0ZV9vb2YKICAgICAgICAgICAgYmxlbmRfcHJlZCA9IDAuNzUgKiBibGVuZF9wcmVkICsgMC4yNSAqIHJhbmswMShwcmVkc1tuYW1lXSkKICAgICAgICAgICAgbWVtYmVycy5hcHBlbmQobmFtZSkKICAgICAgICAgICAgYmVzdF9zY29yZSA9IG1heChiZXN0X3Njb3JlLCBzY29yZSkKICAgIHJldHVybiBibGVuZF9vb2YsIGJsZW5kX3ByZWQsIG1lbWJlcnMsIHJvY19hdWNfc2NvcmUoeSwgYmxlbmRfb29mKQoKCmRlZiBzYXZlX3N1Ym1pc3Npb24oc2FtcGxlLCB0YXJnZXQsIHByZWQsIGZpbGVuYW1lKToKICAgIG91dCA9IHNhbXBsZS5jb3B5KCkKICAgIG91dFt0YXJnZXRdID0gbnAuY2xpcChwcmVkLCAxZS03LCAxIC0gMWUtNykKICAgIG91dC50b19jc3YoZmlsZW5hbWUsIGluZGV4PUZhbHNlKQoKCmRlZiBtYWluKCk6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWZhc3QiLCBhY3Rpb249InN0b3JlX3RydWUiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1mYWxsYmFjayIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgc3RhcnRlZCA9IHRpbWUudGltZSgpCiAgICB0cmFpbiA9IHBkLnJlYWRfY3N2KCJ0cmFpbi5jc3YiKQogICAgdGVzdCA9IHBkLnJlYWRfY3N2KCJ0ZXN0LmNzdiIpCiAgICBzYW1wbGUgPSBwZC5yZWFkX2Nzdigic2FtcGxlX3N1Ym1pc3Npb24uY3N2IikKICAgIHRhcmdldCwgaWRfY29sLCBmZWF0dXJlcyA9IGZpbmRfY29sdW1ucyh0cmFpbiwgdGVzdCwgc2FtcGxlKQogICAgeSwgbWFwcGluZyA9IG5vcm1hbGl6ZV90YXJnZXQodHJhaW5bdGFyZ2V0XSkKICAgIHh0ciwgeHRlLCBjYXRfY29scywgbnVtX2NvbHMgPSBwcmVwYXJlX2ZyYW1lcyh0cmFpbiwgdGVzdCwgZmVhdHVyZXMpCiAgICBuX3NwbGl0cyA9IDMgaWYgKGFyZ3MuZmFzdCBvciBsZW4odHJhaW4pID4gMzAwMDApIGVsc2UgNAogICAgZm9sZHMgPSBsaXN0KFN0cmF0aWZpZWRLRm9sZChuX3NwbGl0cz1uX3NwbGl0cywgc2h1ZmZsZT1UcnVlLCByYW5kb21fc3RhdGU9U0VFRCkuc3BsaXQoeHRyLCB5KSkKICAgIG1vZGVscyA9IHNrbGVhcm5fbW9kZWxzKGNhdF9jb2xzLCBudW1fY29scywgbGVuKHRyYWluKSwgZmFsbGJhY2s9YXJncy5mYWxsYmFjaykKICAgIGlmIG5vdCBhcmdzLmZhbGxiYWNrOgogICAgICAgIGFkZF9ib29zdGVycyhtb2RlbHMsIGNhdF9jb2xzLCBsZW4odHJhaW4pLCBhcmdzLmZhc3QpCiAgICBwcmludChqc29uLmR1bXBzKHsicm93cyI6IGxlbih0cmFpbiksICJ0ZXN0X3Jvd3MiOiBsZW4odGVzdCksICJmZWF0dXJlcyI6IGxlbihmZWF0dXJlcyksCiAgICAgICAgICAgICAgICAgICAgICAiY2F0ZWdvcmljYWwiOiBsZW4oY2F0X2NvbHMpLCAibnVtZXJpYyI6IGxlbihudW1fY29scyksICJmb2xkcyI6IG5fc3BsaXRzLAogICAgICAgICAgICAgICAgICAgICAgIm1vZGVscyI6IGxpc3QobW9kZWxzKX0sIHNvcnRfa2V5cz1UcnVlKSkKICAgIG9vZnMsIHByZWRzLCByZXN1bHRzID0ge30sIHt9LCBbXQogICAgZm9yIG5hbWUsIG1vZGVsIGluIG1vZGVscy5pdGVtcygpOgogICAgICAgIHRyeToKICAgICAgICAgICAgdDAgPSB0aW1lLnRpbWUoKQogICAgICAgICAgICBvb2YsIHByZWQsIGZvbGRfc2NvcmVzID0gZml0X3ByZWRpY3RfbW9kZWwobmFtZSwgbW9kZWwsIHh0ciwgeHRlLCB5LCBmb2xkcywgY2F0X2NvbHMpCiAgICAgICAgICAgIHNjb3JlID0gcm9jX2F1Y19zY29yZSh5LCBvb2YpCiAgICAgICAgICAgIG9vZnNbbmFtZV0sIHByZWRzW25hbWVdID0gb29mLCBwcmVkCiAgICAgICAgICAgIHJlc3VsdHMuYXBwZW5kKHsibmFtZSI6IG5hbWUsICJjdl9hdWMiOiBzY29yZSwgImZvbGRfYXVjIjogZm9sZF9zY29yZXMsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAic2Vjb25kcyI6IHJvdW5kKHRpbWUudGltZSgpIC0gdDAsIDEpfSkKICAgICAgICAgICAgcHJpbnQoZiJNT0RFTCB7bmFtZX0gY3ZfYXVjPXtzY29yZTouNmZ9IGZvbGRzPXsnLCcuam9pbihmJ3tzOi41Zn0nIGZvciBzIGluIGZvbGRfc2NvcmVzKX0iKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOgogICAgICAgICAgICBwcmludChmIk1PREVMX0ZBSUxFRCB7bmFtZX06IHt0eXBlKGV4YykuX19uYW1lX199OiB7ZXhjfSIpCiAgICBpZiBub3QgcmVzdWx0czoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkFsbCBtb2RlbHMgZmFpbGVkIikKICAgIHJlc3VsdHMuc29ydChrZXk9bGFtYmRhIHI6IHJbImN2X2F1YyJdLCByZXZlcnNlPVRydWUpCiAgICBuYW1lcyA9IFtyWyJuYW1lIl0gZm9yIHIgaW4gcmVzdWx0c10KICAgIF8sIGJsZW5kX3ByZWQsIG1lbWJlcnMsIGJsZW5kX3Njb3JlID0gZ3JlZWR5X2JsZW5kKG9vZnMsIHByZWRzLCB5LCBuYW1lcykKICAgIGNhbmRpZGF0ZXMgPSBbKCJibGVuZCIsIGJsZW5kX3ByZWQsIGJsZW5kX3Njb3JlLCBtZW1iZXJzKV0KICAgIGZvciBpdGVtIGluIHJlc3VsdHM6CiAgICAgICAgY2FuZGlkYXRlcy5hcHBlbmQoKGl0ZW1bIm5hbWUiXSwgcmFuazAxKHByZWRzW2l0ZW1bIm5hbWUiXV0pLCBpdGVtWyJjdl9hdWMiXSwgW2l0ZW1bIm5hbWUiXV0pKQogICAgIyBBIHN0YWJsZSBicm9hZCBhdmVyYWdlIGlzIHVzZWZ1bCB3aGVuIENWIGlzIG5vaXN5IG9uIHRpbnkgZGF0YXNldHMuCiAgICB0b3AgPSBuYW1lc1s6IG1pbigzLCBsZW4obmFtZXMpKV0KICAgIGJyb2FkID0gbnAubWVhbihbcmFuazAxKHByZWRzW25dKSBmb3IgbiBpbiB0b3BdLCBheGlzPTApCiAgICBicm9hZF9vb2YgPSBucC5tZWFuKFtyYW5rMDEob29mc1tuXSkgZm9yIG4gaW4gdG9wXSwgYXhpcz0wKQogICAgY2FuZGlkYXRlcy5hcHBlbmQoKCJicm9hZF9ibGVuZCIsIGJyb2FkLCByb2NfYXVjX3Njb3JlKHksIGJyb2FkX29vZiksIHRvcCkpCiAgICBjYW5kaWRhdGVzLnNvcnQoa2V5PWxhbWJkYSB4OiB4WzJdLCByZXZlcnNlPVRydWUpCiAgICBmaWxlcywgc2VlbiA9IFtdLCBbXQogICAgZm9yIGlkeCwgKG5hbWUsIHByZWQsIHNjb3JlLCBtZW1iZXJzKSBpbiBlbnVtZXJhdGUoY2FuZGlkYXRlcyk6CiAgICAgICAgaWYgYW55KG5wLmNvcnJjb2VmKHByZWQsIHApWzAsIDFdID4gMC45OTk5OCBmb3IgcCBpbiBzZWVuKToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBmaWxlbmFtZSA9IGYiY2FuZGlkYXRlX3tsZW4oZmlsZXMpKzE6MDJkfV97bmFtZX0uY3N2IgogICAgICAgIHNhdmVfc3VibWlzc2lvbihzYW1wbGUsIHRhcmdldCwgcHJlZCwgZmlsZW5hbWUpCiAgICAgICAgZGl2ZXJzaXR5ID0gMS4wIGlmIG5vdCBzZWVuIGVsc2UgZmxvYXQoMSAtIG1heChucC5jb3JyY29lZihwcmVkLCBwKVswLCAxXSBmb3IgcCBpbiBzZWVuKSkKICAgICAgICBmaWxlcy5hcHBlbmQoeyJmaWxlIjogZmlsZW5hbWUsICJuYW1lIjogbmFtZSwgImN2X2F1YyI6IHNjb3JlLAogICAgICAgICAgICAgICAgICAgICAgIm1lbWJlcnMiOiBtZW1iZXJzLCAiZGl2ZXJzaXR5X2Zyb21fZWFybGllciI6IGRpdmVyc2l0eX0pCiAgICAgICAgc2Vlbi5hcHBlbmQocHJlZCkKICAgICAgICBpZiBsZW4oZmlsZXMpID49IDg6CiAgICAgICAgICAgIGJyZWFrCiAgICBtYW5pZmVzdCA9IHsKICAgICAgICAic2NoZW1hIjogeyJ0YXJnZXQiOiB0YXJnZXQsICJpZCI6IGlkX2NvbCwgImZlYXR1cmVzIjogbGVuKGZlYXR1cmVzKSwKICAgICAgICAgICAgICAgICAgICJjYXRlZ29yaWNhbCI6IGNhdF9jb2xzLCAibnVtZXJpYyI6IG51bV9jb2xzLCAidGFyZ2V0X21hcHBpbmciOiB7c3RyKGspOiB2IGZvciBrLCB2IGluIG1hcHBpbmcuaXRlbXMoKX19LAogICAgICAgICJtb2RlbHMiOiByZXN1bHRzLCAiY2FuZGlkYXRlcyI6IGZpbGVzLAogICAgICAgICJyZWNvbW1lbmRlZF9kaXZlcnNlX2ZpbGUiOiBmaWxlc1sxXVsiZmlsZSJdIGlmIGxlbihmaWxlcykgPiAxIGVsc2UgZmlsZXNbMF1bImZpbGUiXSwKICAgICAgICAiZWxhcHNlZF9zZWNvbmRzIjogcm91bmQodGltZS50aW1lKCkgLSBzdGFydGVkLCAxKSwgInNlZWQiOiBTRUVELAogICAgfQogICAgUGF0aCgiYXV0b21sX21hbmlmZXN0Lmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMobWFuaWZlc3QsIGluZGVudD0yKSwgZW5jb2Rpbmc9InV0Zi04IikKICAgIHByaW50KCJDQU5ESURBVEVTICIgKyAiICIuam9pbihpdGVtWyJmaWxlIl0gZm9yIGl0ZW0gaW4gZmlsZXMpKQogICAgcHJpbnQoZiJET05FIGVsYXBzZWRfc2Vjb25kcz17bWFuaWZlc3RbJ2VsYXBzZWRfc2Vjb25kcyddfSIpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=\"}")
work = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
agent_dir = work / 'agent'
if agent_dir.exists():
    shutil.rmtree(agent_dir)
agent_dir.mkdir(parents=True)
for relative, encoded in FILES.items():
    destination = agent_dir / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(base64.b64decode(encoded))
print(f'Restored {len(FILES)} files to {agent_dir}')

In [ ]:
zip_path = work / 'submission.zip'
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(agent_dir.rglob('*')):
        if path.is_file():
            archive.write(path, path.relative_to(agent_dir).as_posix())
with zipfile.ZipFile(zip_path) as archive:
    names = archive.namelist()
assert 'agent.yaml' in names and all(not n.startswith('agent/') for n in names)
print(f'Created {zip_path} ({zip_path.stat().st_size:,} bytes)')
print('\n'.join(names))

The notebook output named `submission.zip` is the artifact to submit to the competition.